In [ ]:
import pandas as pd


from autoslo.slo.slo_resolver import SloResolver
from autoslo.config.component_configs import SloResolverConfig
from logos import Logos

from autoslo.filesystem.logos_export import logos_df

import autoslo.filesystem.path_utils as pu
from autoslo.filesystem.yaml_helpers import load_yaml

In [ ]:
run_id = "1783260865180"

exec_cfg = load_yaml(pu.get_runs_dir() / str(run_id) / "execution_config.yml")


resolver = SloResolver(
    SloResolverConfig(
        slo_s=exec_cfg["slo_resolver_config"]["slo_s"],
        slo_dict_filename=exec_cfg["slo_resolver_config"]["slo_dict_filename"],
    )
)
df = logos_df(run_id=run_id, slo_resolver=resolver, include_named_query_features=True)

Loading model checkpoint from /home/markakis/chunkbench/data/iconq_models/model_v8/model_1782160265.pth


In [8]:
df.head()

,wall_clock_s,rel_time_s,source,event_type,query_id,query_text_id,cluster_name,reason,workload_name,num_queries,...,item#cardinality,time_dim#cardinality,date_dim#cardinality,catalog_page#cardinality,household_demographics#cardinality,web_page#cardinality,promotion#cardinality,store#cardinality,reason#cardinality,web_site#cardinality
0,1.783261e+09,0.000000,ManagedClusterPool,spin_up_requested,None,None,,initial,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.783261e+09,0.000000,RedshiftServerlessProvisioner,spin_up_started,None,None,autoslo-32-1783260865180-0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.783261e+09,0.000000,ManagedClusterPool,cluster_ready,None,None,autoslo-32-1783260865180-0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.783261e+09,-29.999977,WorkloadRunner,run_start,None,None,,NaN,redbench_provisioned_157_0,1392.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.783261e+09,0.030367,WorkloadRunner,arrival,204510,ext_tpcds1000#009#003,,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693152,0.0


In [9]:
per_unit = [
    "slo_violated",
    "slo_s",
    "slo_overshoot_s",
    "relative_violation",
    "final_latency_s",
    "actual_execution_latency_s",
    "selected_cluster_name",
    "selected_rpu",
    "prediction_error",
] + [df_col for df_col in df.columns if "#" in df_col]

In [10]:
lg = Logos.from_parsed_table(
    data=df,
    workdir='.',
    source_id="structured_log_p2",
    template_col="event_type",
    passthrough_cols=["wall_clock_s", "rel_time_s", "query_id", "query_text_id"],
    per_unit_cols=per_unit,
)
lg.set_causal_unit("query_id")
lg.prepare(default_imp="zero_imp", force=True)

Imputing missing values...:   0%|          | 0/282 [00:00<?, ?it/s]

One-hot encoding categorical variables...:   0%|          | 0/21 [00:00<?, ?it/s]

In [11]:
lg.prepared_variables

,Name,Base,Pre-agg Value,Agg,Post-agg Value,Tag,Base Variable Occurences,Type,Examples,From regex,TemplateText
0,ebdcf81f_2+mean,ebdcf81f_2,,mean,,routing_score latency_s_for_routing mean,10144,num,"[15.494, 17.444, 17.443, 23.051, 26.054]",False,routing_score
1,ebdcf81f_3+mean,ebdcf81f_3,,mean,,routing_score slo_violation mean,10144,num,"[0.0, 0.5, 0.167, 0.143, 0.25]",False,routing_score
2,ebdcf81f_4+mean,ebdcf81f_4,,mean,,routing_score cost mean,10144,num,"[0.0, 0.2, 0.4, 0.6, 0.8]",False,routing_score
3,ebdcf81f_5+mean,ebdcf81f_5,,mean,,routing_score cache_risk mean,10144,num,[0.0],False,routing_score
4,ef3b209b_2+mean,ef3b209b_2,,mean,,routing latency_s_for_routing mean,1392,num,"[15.494, 17.444, 17.443, 23.051, 26.054]",False,routing
...,...,...,...,...,...,...,...,...,...,...,...
242,ed4cac1d_0+last=autoslo-16-1783260865180-8,ed4cac1d_0,,last,autoslo-16-1783260865180-8,selected_cluster_name last autoslo-16-17832608...,21236,str,"[autoslo-32-1783260865180-0, autoslo-32-178326...",True,
243,ed4cac1d_0+last=autoslo-16-1783260865180-9,ed4cac1d_0,,last,autoslo-16-1783260865180-9,selected_cluster_name last autoslo-16-17832608...,21236,str,"[autoslo-32-1783260865180-0, autoslo-32-178326...",True,
244,ed4cac1d_0+last=autoslo-32-1783260865180-0,ed4cac1d_0,,last,autoslo-32-1783260865180-0,selected_cluster_name last autoslo-32-17832608...,21236,str,"[autoslo-32-1783260865180-0, autoslo-32-178326...",True,
245,ed4cac1d_0+last=autoslo-32-1783260865180-1,ed4cac1d_0,,last,autoslo-32-1783260865180-1,selected_cluster_name last autoslo-32-17832608...,21236,str,"[autoslo-32-1783260865180-0, autoslo-32-178326...",True,


In [15]:
lg.rank_candidate_causes("slo_violated mean", prune_candidates=True)

,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,17060c6a_0+mean,prediction_error mean,slo_violated mean,0.001491,2.065270e-82,Undecided,Rejected


In [13]:
lg.accept("prediction_error mean", "slo_violated mean")


(np.float64(0.25191815856777494), '17060c6a_0+mean')

In [14]:
lg.rank_candidate_causes("prediction_error mean")

,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,86a6fe79_0+mean,final_latency_s mean,prediction_error mean,0.989131,0.000000,Undecided,Undecided
1,c1357543_0+mean,slo_overshoot_s mean,prediction_error mean,1.076566,0.000000,Undecided,Undecided
2,ef3b209b_2+mean,routing latency_s_for_routing mean,prediction_error mean,-0.126342,0.597646,Undecided,Undecided
3,133bdb9d_2+mean,query_routed predicted_latency_s mean,prediction_error mean,-0.126340,0.597652,Undecided,Undecided


In [15]:
lg.accept("final_latency_s mean", "prediction_error mean")


(np.float64(0.3333333333333333), '86a6fe79_0+mean')

In [16]:
lg.rank_candidate_causes("final_latency_s mean")

,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,c1357543_0+mean,slo_overshoot_s mean,final_latency_s mean,1.073506,0.0,Undecided,Undecided
1,0b37e060_0+mean,actual_execution_latency_s mean,final_latency_s mean,1.000015,0.0,Undecided,Undecided


In [17]:
lg.accept("actual_execution_latency_s mean", "final_latency_s mean")


(np.float64(0.3759640102827764), '0b37e060_0+mean')

In [18]:
lg.rank_candidate_causes("actual_execution_latency_s mean")

,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,c1357543_0+mean,slo_overshoot_s mean,actual_execution_latency_s mean,1.073495,0.0,Undecided,Undecided
1,86a6fe79_0+mean,final_latency_s mean,actual_execution_latency_s mean,0.999985,0.0,Rejected,Accepted
